# 从零实现 GraphCL：图增强、GIN 编码器与多正例 NT-Xent

本 Notebook 只使用 PyTorch 张量和 `nn.Module`，手写图批处理合同、节点/边/特征增强、GIN 消息传递、图级池化、投影头与对称 NT-Xent；不使用 PyG、DGL、现成 GNN、`MultiheadAttention` 或 Transformer。重点不是“跑出一个好看的准确率”，而是让随机增强、跨图隔离、假负例与发布边界都能被断言审计。

原始思想参考：[GraphCL, NeurIPS 2020](https://arxiv.org/abs/2010.13902)、[GIN, ICLR 2019](https://arxiv.org/abs/1810.00826)、[SimCLR, ICML 2020](https://arxiv.org/abs/2002.05709)。这里使用离线合成小图做受控实验，结果只证明实现链路在该 fixture 上有效，不代表真实分子图或社交图泛化。


In [ ]:
from __future__ import annotations  # 导入本单元所需的依赖。

import copy, hashlib, json, math, random, warnings  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message=".*pynvml.*", category=FutureWarning)  # 计算并保存当前步骤的中间状态。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED = 6101  # 计算并保存当前步骤的中间状态。
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)  # 执行当前语句以推进本节示例。
torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE = torch.device("cpu")  # 计算并保存当前步骤的中间状态。

def canonical_digest(payload) -> str:  # 定义本节可复用的核心函数。
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))  # 计算并保存当前步骤的中间状态。
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()  # 返回当前分支计算出的结果。

def state_descriptor(state: dict[str, torch.Tensor]) -> dict:  # 定义本节可复用的核心函数。
    result = {}  # 计算并保存当前步骤的中间状态。
    for key in sorted(state):  # 遍历输入元素以累积或检查结果。
        value = state[key].detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
        result[key] = {  # 计算并保存当前步骤的中间状态。
            "dtype": str(value.dtype), "shape": list(value.shape),  # 执行当前语句以推进本节示例。
            "sha256": hashlib.sha256(value.numpy().tobytes()).hexdigest(),  # 执行当前语句以推进本节示例。
        }  # 执行当前语句以推进本节示例。
    return result  # 返回当前分支计算出的结果。

assert DEVICE.type == "cpu" and torch.get_num_threads() == 1  # 用受控断言验证关键不变量。
assert canonical_digest({"b": 2, "a": 1}) == canonical_digest({"a": 1, "b": 2})  # 用受控断言验证关键不变量。
assert state_descriptor({"x": torch.tensor([1.0])}) != state_descriptor({"x": torch.tensor([2.0])})  # 用受控断言验证关键不变量。
assert not any(name in globals() for name in ("torch_geometric", "dgl"))  # 用受控断言验证关键不变量。


## 1. Packed graph batch 是第一道隔离边界

节点特征为 `x:[N,F]`，边为 `edge_index:[2,E]`，`batch:[N]` 指明每个节点属于哪张图。任何边都必须满足 `batch[src] == batch[dst]`；否则一次消息传递就会把测试样本或其他租户的信息混进来。这里不使用 padding，空图被拒绝，图编号必须从 0 连续出现。

对图置换只改变节点局部顺序，不应改变图级表示。`semantic()` 同时记录特征值、边、batch、dtype 与 shape，后面发布包装器会重新计算，而不是相信调用者声称的摘要。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class GraphBatch:  # 定义承载本节状态与行为的数据结构。
    x: torch.Tensor  # 执行当前语句以推进本节示例。
    edge_index: torch.Tensor  # 执行当前语句以推进本节示例。
    batch: torch.Tensor  # 执行当前语句以推进本节示例。

    def validate(self) -> "GraphBatch":  # 定义本节可复用的核心函数。
        if self.x.ndim != 2 or self.x.numel() == 0 or not torch.isfinite(self.x).all():  # 按当前条件选择后续控制路径。
            raise ValueError("x 必须是有限的非空 [N,F]")  # 遇到非法合同立即显式失败。
        if self.edge_index.ndim != 2 or self.edge_index.shape[0] != 2 or self.edge_index.dtype != torch.long:  # 按当前条件选择后续控制路径。
            raise ValueError("edge_index 必须是 long [2,E]")  # 遇到非法合同立即显式失败。
        if self.batch.shape != (self.x.shape[0],) or self.batch.dtype != torch.long:  # 按当前条件选择后续控制路径。
            raise ValueError("batch 必须是 long [N]")  # 遇到非法合同立即显式失败。
        if self.batch.numel() and int(self.batch.min()) != 0:  # 按当前条件选择后续控制路径。
            raise ValueError("图编号必须从 0 开始")  # 遇到非法合同立即显式失败。
        ids = self.batch.unique(sorted=True)  # 计算并保存当前步骤的中间状态。
        if not torch.equal(ids, torch.arange(len(ids), device=ids.device)):  # 按当前条件选择后续控制路径。
            raise ValueError("图编号必须连续且每图至少一个节点")  # 遇到非法合同立即显式失败。
        if self.edge_index.numel():  # 按当前条件选择后续控制路径。
            src, dst = self.edge_index  # 计算并保存当前步骤的中间状态。
            if int(src.min()) < 0 or int(dst.min()) < 0 or int(src.max()) >= len(self.x) or int(dst.max()) >= len(self.x):  # 按当前条件选择后续控制路径。
                raise ValueError("边索引越界")  # 遇到非法合同立即显式失败。
            if not torch.equal(self.batch[src], self.batch[dst]):  # 按当前条件选择后续控制路径。
                raise ValueError("检测到跨图边")  # 遇到非法合同立即显式失败。
        return self  # 返回当前分支计算出的结果。

    @property  # 为下方定义附加声明式配置。
    def num_graphs(self) -> int:  # 定义本节可复用的核心函数。
        return int(self.batch.max()) + 1  # 返回当前分支计算出的结果。

    def semantic(self) -> dict:  # 定义本节可复用的核心函数。
        self.validate()  # 执行当前语句以推进本节示例。
        def desc(t):  # 定义本节可复用的核心函数。
            t = t.detach().cpu().contiguous()  # 计算并保存当前步骤的中间状态。
            return {"dtype": str(t.dtype), "shape": list(t.shape),  # 返回当前分支计算出的结果。
                    "sha256": hashlib.sha256(t.numpy().tobytes()).hexdigest()}  # 执行当前语句以推进本节示例。
        return {"x": desc(self.x), "edge_index": desc(self.edge_index), "batch": desc(self.batch)}  # 返回当前分支计算出的结果。

probe = GraphBatch(torch.eye(3), torch.tensor([[0,1,1,2],[1,0,2,1]]), torch.zeros(3, dtype=torch.long)).validate()  # 计算并保存当前步骤的中间状态。
assert probe.num_graphs == 1 and probe.semantic()["x"]["shape"] == [3, 3]  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    GraphBatch(torch.ones(4,2), torch.tensor([[0],[2]]), torch.tensor([0,0,1,1])).validate()  # 执行当前语句以推进本节示例。
    raise AssertionError("跨图边未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "跨图" in str(exc)  # 用受控断言验证关键不变量。


## 2. 三类增强及其随机数合同

GraphCL 的两个 view 必须独立采样，但也必须可复现。所有随机操作都显式接收 `torch.Generator`：

- 节点丢弃：每张图至少保留一个节点，随后重映射边索引；
- 边丢弃：本 Notebook 把输入解释为无向图的双向 arc，因此 `(u,v)` 与 `(v,u)` 必须共享一次 Bernoulli 决策；
- 特征遮蔽：逐元素置零，保持 shape、batch 与边不变。

联合 recipe 写为 `node_drop → paired_edge_drop → feature_mask`。增强概率必须在 `[0,1)`，`0` 是 identity oracle 的合法配置。相同 seed 必须逐位复现，不同 seed 则应产生独立 view；随机种子只是随机流标识，不能充当正例 ID。


In [ ]:
def _rand(shape, generator):  # 定义本节可复用的核心函数。
    return torch.rand(shape, generator=generator)  # 返回当前分支计算出的结果。

def validate_reciprocal_arcs61(edge_index: torch.Tensor) -> None:  # 定义本节可复用的核心函数。
    if edge_index.ndim != 2 or edge_index.shape[0] != 2:  # 按当前条件选择后续控制路径。
        raise ValueError("edge_index 必须为 [2,E]")  # 遇到非法合同立即显式失败。
    pairs = list(zip(edge_index[0].tolist(), edge_index[1].tolist()))  # 计算并保存当前步骤的中间状态。
    if len(pairs) != len(set(pairs)):  # 按当前条件选择后续控制路径。
        raise ValueError("无向消息图不能含重复 arc")  # 遇到非法合同立即显式失败。
    pair_set = set(pairs)  # 计算并保存当前步骤的中间状态。
    if any(u == v or (v, u) not in pair_set for u, v in pairs):  # 按当前条件选择后续控制路径。
        raise ValueError("每条无向边必须由互反 arc 表示且不含自环")  # 遇到非法合同立即显式失败。

def augment_graph(g: GraphBatch, *, node_drop: float, edge_drop: float,  # 定义本节可复用的核心函数。
                  feature_mask: float, generator: torch.Generator) -> GraphBatch:  # 执行当前语句以推进本节示例。
    g.validate(); validate_reciprocal_arcs61(g.edge_index)  # 执行当前语句以推进本节示例。
    rates = (node_drop, edge_drop, feature_mask)  # 计算并保存当前步骤的中间状态。
    if any(not isinstance(v, (int, float)) or not 0 <= v < 1 for v in rates):  # 按当前条件选择后续控制路径。
        raise ValueError("增强率必须位于 [0,1)")  # 遇到非法合同立即显式失败。
    keep = _rand((len(g.x),), generator) >= node_drop  # 计算并保存当前步骤的中间状态。
    for graph_id in range(g.num_graphs):  # 遍历输入元素以累积或检查结果。
        members = torch.where(g.batch == graph_id)[0]  # 计算并保存当前步骤的中间状态。
        if not keep[members].any():  # 按当前条件选择后续控制路径。
            keep[members[int(torch.randint(len(members), (1,), generator=generator))]] = True  # 计算并保存当前步骤的中间状态。
    old_to_new = torch.full((len(g.x),), -1, dtype=torch.long)  # 计算并保存当前步骤的中间状态。
    old_to_new[keep] = torch.arange(int(keep.sum()))  # 计算并保存当前步骤的中间状态。
    if g.edge_index.numel():  # 按当前条件选择后续控制路径。
        src, dst = g.edge_index  # 计算并保存当前步骤的中间状态。
        candidate = torch.where(keep[src] & keep[dst])[0]  # 计算并保存当前步骤的中间状态。
        edge_keep = torch.zeros(g.edge_index.shape[1], dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
        if candidate.numel():  # 按当前条件选择后续控制路径。
            pair_key = torch.minimum(src[candidate], dst[candidate]) * len(g.x) + torch.maximum(src[candidate], dst[candidate])  # 计算并保存当前步骤的中间状态。
            _, inverse = torch.unique(pair_key, sorted=True, return_inverse=True)  # 计算并保存当前步骤的中间状态。
            pair_count = int(inverse.max()) + 1  # 计算并保存当前步骤的中间状态。
            pair_keep = _rand((pair_count,), generator) >= edge_drop  # 计算并保存当前步骤的中间状态。
            edge_keep[candidate] = pair_keep[inverse]  # 计算并保存当前步骤的中间状态。
        edge_index = old_to_new[g.edge_index[:, edge_keep]]  # 计算并保存当前步骤的中间状态。
    else:  # 处理前置条件不成立的分支。
        edge_index = g.edge_index.clone()  # 计算并保存当前步骤的中间状态。
    x = g.x[keep].clone()  # 计算并保存当前步骤的中间状态。
    x[_rand(x.shape, generator) < feature_mask] = 0.0  # 计算并保存当前步骤的中间状态。
    augmented = GraphBatch(x, edge_index, g.batch[keep].clone()).validate()  # 计算并保存当前步骤的中间状态。
    validate_reciprocal_arcs61(augmented.edge_index)  # 执行当前语句以推进本节示例。
    return augmented  # 返回当前分支计算出的结果。

gen_a = torch.Generator().manual_seed(7)  # 计算并保存当前步骤的中间状态。
identity = augment_graph(probe, node_drop=0, edge_drop=0, feature_mask=0, generator=gen_a)  # 计算并保存当前步骤的中间状态。
assert torch.equal(identity.x, probe.x) and torch.equal(identity.edge_index, probe.edge_index)  # 用受控断言验证关键不变量。
v1 = augment_graph(probe, node_drop=.4, edge_drop=.3, feature_mask=.2, generator=torch.Generator().manual_seed(8))  # 计算并保存当前步骤的中间状态。
v2 = augment_graph(probe, node_drop=.4, edge_drop=.3, feature_mask=.2, generator=torch.Generator().manual_seed(8))  # 计算并保存当前步骤的中间状态。
assert v1.semantic() == v2.semantic() and v1.x.shape[0] >= 1  # 用受控断言验证关键不变量。
assert torch.equal(v1.batch[v1.edge_index[0]], v1.batch[v1.edge_index[1]])  # 用受控断言验证关键不变量。
view_seed_a = augment_graph(probe, node_drop=0, edge_drop=0, feature_mask=.5, generator=torch.Generator().manual_seed(101))  # 计算并保存当前步骤的中间状态。
view_seed_b = augment_graph(probe, node_drop=0, edge_drop=0, feature_mask=.5, generator=torch.Generator().manual_seed(102))  # 计算并保存当前步骤的中间状态。
assert not torch.equal(view_seed_a.x, view_seed_b.x)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    validate_reciprocal_arcs61(torch.tensor([[0, 1], [1, 2]]))  # 执行当前语句以推进本节示例。
    raise AssertionError("缺失反向 arc 未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "互反" in str(exc)  # 用受控断言验证关键不变量。


## 3. 手写 GIN 层与图级池化

对节点 $v$，GIN 更新为

$$h_v^{(k+1)}=\mathrm{MLP}_k\left((1+\epsilon_k)h_v^{(k)}+\sum_{u\in\mathcal N(v)}h_u^{(k)}\right).$$

`index_add_` 显式完成邻居求和，复杂度为 $O(EH+NH^2)$，内存为 $O(NH+EH)$。图表示对各层节点表示做 mean pooling 后拼接，避免大图仅因节点数更大而获得更大范数。为了让置换 oracle 严格成立，层中不放依赖 batch 统计的 BatchNorm。


In [ ]:
def mean_pool(x: torch.Tensor, batch: torch.Tensor, num_graphs: int) -> torch.Tensor:  # 定义本节可复用的核心函数。
    out = x.new_zeros((num_graphs, x.shape[1]))  # 计算并保存当前步骤的中间状态。
    out.index_add_(0, batch, x)  # 执行当前语句以推进本节示例。
    count = torch.bincount(batch, minlength=num_graphs).clamp_min(1).to(x.dtype).unsqueeze(1)  # 计算并保存当前步骤的中间状态。
    return out / count  # 返回当前分支计算出的结果。

class GINLayer(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.eps = nn.Parameter(torch.zeros(()))  # 计算并保存当前步骤的中间状态。
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))  # 计算并保存当前步骤的中间状态。

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        if x.ndim != 2 or edge_index.ndim != 2 or edge_index.shape[0] != 2:  # 按当前条件选择后续控制路径。
            raise ValueError("GIN 输入 shape 非法")  # 遇到非法合同立即显式失败。
        agg = torch.zeros_like(x)  # 计算并保存当前步骤的中间状态。
        if edge_index.numel():  # 按当前条件选择后续控制路径。
            src, dst = edge_index  # 计算并保存当前步骤的中间状态。
            agg.index_add_(0, dst, x[src])  # 执行当前语句以推进本节示例。
        return self.mlp((1.0 + self.eps) * x + agg)  # 返回当前分支计算出的结果。

class GINEncoder(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_dim: int, hidden: int, depth: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.input = nn.Linear(in_dim, hidden)  # 计算并保存当前步骤的中间状态。
        self.layers = nn.ModuleList([GINLayer(hidden) for _ in range(depth)])  # 计算并保存当前步骤的中间状态。
        self.out_dim = hidden * (depth + 1)  # 计算并保存当前步骤的中间状态。

    def forward(self, graph: GraphBatch) -> torch.Tensor:  # 定义本节可复用的核心函数。
        graph.validate()  # 执行当前语句以推进本节示例。
        h = F.relu(self.input(graph.x))  # 计算并保存当前步骤的中间状态。
        pooled = [mean_pool(h, graph.batch, graph.num_graphs)]  # 计算并保存当前步骤的中间状态。
        for layer in self.layers:  # 遍历输入元素以累积或检查结果。
            h = F.relu(layer(h, graph.edge_index))  # 计算并保存当前步骤的中间状态。
            pooled.append(mean_pool(h, graph.batch, graph.num_graphs))  # 执行当前语句以推进本节示例。
        return torch.cat(pooled, dim=-1)  # 返回当前分支计算出的结果。

class ProjectionHead(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim: int, proj_dim: int):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, proj_dim))  # 计算并保存当前步骤的中间状态。
    def forward(self, graph_embedding: torch.Tensor) -> torch.Tensor:  # 定义本节可复用的核心函数。
        return F.normalize(self.net(graph_embedding), dim=-1)  # 返回当前分支计算出的结果。

class GraphCL(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, in_dim=3, hidden=16, depth=2, proj_dim=12):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.encoder = GINEncoder(in_dim, hidden, depth)  # 计算并保存当前步骤的中间状态。
        self.projector = ProjectionHead(self.encoder.out_dim, proj_dim)  # 计算并保存当前步骤的中间状态。
    def forward(self, graph: GraphBatch, project: bool = True) -> torch.Tensor:  # 定义本节可复用的核心函数。
        h = self.encoder(graph)  # 计算并保存当前步骤的中间状态。
        return self.projector(h) if project else h  # 返回当前分支计算出的结果。

model61 = GraphCL()  # 计算并保存当前步骤的中间状态。
assert model61(probe).shape == (1, 12)  # 用受控断言验证关键不变量。
assert torch.allclose(model61(probe).norm(dim=1), torch.ones(1), atol=1e-6)  # 用受控断言验证关键不变量。


## 4. 节点置换与 view identity oracle

消息传递应对节点置换等变，图级池化应对置换不变。测试不能只比较 shape：下面同时重排特征、batch，并通过逆映射改写边索引，再比较数值。identity view 则验证增强率全零不会暗中重排或修改输入。


In [ ]:
def permute_graph(g: GraphBatch, perm: torch.Tensor) -> GraphBatch:  # 定义本节可复用的核心函数。
    if sorted(perm.tolist()) != list(range(len(g.x))):  # 按当前条件选择后续控制路径。
        raise ValueError("perm 不是合法置换")  # 遇到非法合同立即显式失败。
    inv = torch.empty_like(perm); inv[perm] = torch.arange(len(perm))  # 计算并保存当前步骤的中间状态。
    return GraphBatch(g.x[perm], inv[g.edge_index], g.batch[perm]).validate()  # 返回当前分支计算出的结果。

square = GraphBatch(  # 计算并保存当前步骤的中间状态。
    torch.tensor([[1.,0,0],[0,1,0],[0,0,1],[1,1,0]]),  # 执行当前语句以推进本节示例。
    torch.tensor([[0,1,1,2,2,3,3,0],[1,0,2,1,3,2,0,3]]),  # 执行当前语句以推进本节示例。
    torch.zeros(4, dtype=torch.long),  # 计算并保存当前步骤的中间状态。
).validate()  # 执行当前语句以推进本节示例。
perm = torch.tensor([2,0,3,1])  # 计算并保存当前步骤的中间状态。
model61.eval()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    base = model61(square, project=False)  # 计算并保存当前步骤的中间状态。
    moved = model61(permute_graph(square, perm), project=False)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(base, moved, atol=1e-6)  # 用受控断言验证关键不变量。
assert identity.semantic() == probe.semantic()  # 用受控断言验证关键不变量。


## 5. 对称、多正例 NT-Xent 与假负例边界

把两批 view 拼成 $Z=[Z^{(1)};Z^{(2)}]$。同一 `group_id` 的其他样本都是正例，自己从分母剔除：

$$\ell_i=-\log\frac{\sum_{p\in P(i)}\exp(s_{ip}/\tau)}{\sum_{a\ne i}\exp(s_{ia}/\tau)}.$$

这比“每行只有固定配对列”的实现更安全：若 batch 中同一原图出现多个增强，或数据去重失败，额外副本不应被当成假负例。仍需注意：不知道语义标签时，不同 ID 也可能语义相同；生产系统应做近重复聚类、采样审计或 debiased contrastive loss。


In [ ]:
def multi_positive_nt_xent(z1: torch.Tensor, z2: torch.Tensor,  # 定义本节可复用的核心函数。
                           group_ids: torch.Tensor, temperature: float) -> torch.Tensor:  # 执行当前语句以推进本节示例。
    if z1.shape != z2.shape or z1.ndim != 2 or group_ids.shape != (len(z1),):  # 按当前条件选择后续控制路径。
        raise ValueError("NT-Xent shape 合同不匹配")  # 遇到非法合同立即显式失败。
    if not 0 < temperature <= 2 or not torch.isfinite(z1).all() or not torch.isfinite(z2).all():  # 按当前条件选择后续控制路径。
        raise ValueError("temperature/embedding 非法")  # 遇到非法合同立即显式失败。
    z = F.normalize(torch.cat([z1, z2]), dim=-1)  # 计算并保存当前步骤的中间状态。
    ids = torch.cat([group_ids, group_ids])  # 计算并保存当前步骤的中间状态。
    logits = z @ z.T / temperature  # 计算并保存当前步骤的中间状态。
    eye = torch.eye(len(z), dtype=torch.bool)  # 计算并保存当前步骤的中间状态。
    positive = ids[:, None].eq(ids[None, :]) & ~eye  # 计算并保存当前步骤的中间状态。
    if not positive.any(dim=1).all():  # 按当前条件选择后续控制路径。
        raise ValueError("每个 anchor 至少需要一个正例")  # 遇到非法合同立即显式失败。
    logits = logits.masked_fill(eye, -torch.inf)  # 计算并保存当前步骤的中间状态。
    log_num = torch.logsumexp(logits.masked_fill(~positive, -torch.inf), dim=1)  # 计算并保存当前步骤的中间状态。
    log_den = torch.logsumexp(logits, dim=1)  # 计算并保存当前步骤的中间状态。
    return (log_den - log_num).mean()  # 返回当前分支计算出的结果。

z_a = torch.tensor([[1.,0.],[0.,1.]])  # 计算并保存当前步骤的中间状态。
z_b = torch.tensor([[1.,0.],[0.,1.]])  # 计算并保存当前步骤的中间状态。
ids = torch.tensor([10,20])  # 计算并保存当前步骤的中间状态。
loss_ab = multi_positive_nt_xent(z_a, z_b, ids, .5)  # 计算并保存当前步骤的中间状态。
loss_ba = multi_positive_nt_xent(z_b, z_a, ids, .5)  # 计算并保存当前步骤的中间状态。
expected = math.log1p(2 * math.exp(-2.0))  # 计算并保存当前步骤的中间状态。
assert torch.allclose(loss_ab, torch.tensor(expected), atol=1e-6)  # 用受控断言验证关键不变量。
assert torch.allclose(loss_ab, loss_ba, atol=1e-7)  # 用受控断言验证关键不变量。
assert multi_positive_nt_xent(z_a, z_b, ids, .25) < loss_ab  # 更尖锐且正例占优

dup_z = torch.tensor([[1.,0.],[1.,0.],[0.,1.]])  # 计算并保存当前步骤的中间状态。
multi = multi_positive_nt_xent(dup_z, dup_z, torch.tensor([1,1,2]), .5)  # 计算并保存当前步骤的中间状态。
false_negative = multi_positive_nt_xent(dup_z, dup_z, torch.tensor([1,9,2]), .5)  # 计算并保存当前步骤的中间状态。
assert multi < false_negative  # 用受控断言验证关键不变量。


## 6. 无标签直通、无跨 split 重复的受控图数据

合成任务仍用两类拓扑做 smoke test：类别 0 是环，类别 1 是星形。但节点特征由 `variant` 的独立随机流生成，与 `label` 无关；同一 variant 换标签时特征逐位相同，只有边结构改变。因此线性头只能利用 encoder 从拓扑与通用特征中学到的表示，不能读取 one-hot 标签通道。

每个原图用完整 tensor 语义摘要生成 `group_id`，两个增强 view 共享该 ID；不同原图即使标签相同也不是正例。train/test 使用不相交 variant 和独立随机流，并显式断言所有原图摘要跨 split 去重。线性评估冻结 encoder，只训练新线性头。

这是小型合成任务；真实评估仍应按 scaffold、主体、时间或来源分组，并在生成增强前完成 split，避免同一语义对象以不同序列化形式跨 split 出现。


In [ ]:
def make_features61(variant: int, n: int) -> torch.Tensor:  # 定义本节可复用的核心函数。
    generator = torch.Generator().manual_seed(SEED + 1000 + int(variant))  # 计算并保存当前步骤的中间状态。
    position = torch.linspace(-1.0, 1.0, n)  # 计算并保存当前步骤的中间状态。
    return torch.stack([  # 返回当前分支计算出的结果。
        torch.ones(n),  # 执行当前语句以推进本节示例。
        .12 * position + .025 * torch.randn(n, generator=generator),  # 计算并保存当前步骤的中间状态。
        .05 * torch.randn(n, generator=generator),  # 计算并保存当前步骤的中间状态。
    ], dim=1)  # 计算并保存当前步骤的中间状态。

def make_graph(label: int, variant: int) -> GraphBatch:  # 定义本节可复用的核心函数。
    if label not in (0, 1) or not isinstance(variant, int) or variant < 0:  # 按当前条件选择后续控制路径。
        raise ValueError("label/variant 合同非法")  # 遇到非法合同立即显式失败。
    n = 6 + variant % 3  # 计算并保存当前步骤的中间状态。
    x = make_features61(variant, n)  # 特征 recipe 不读取 label
    pairs = [(i, (i + 1) % n) for i in range(n)] if label == 0 else [(0, i) for i in range(1, n)]  # 计算并保存当前步骤的中间状态。
    arcs = [(u, v) for u, v in pairs for (u, v) in ((u, v), (v, u))]  # 计算并保存当前步骤的中间状态。
    edge = torch.tensor(arcs, dtype=torch.long).T  # 计算并保存当前步骤的中间状态。
    validate_reciprocal_arcs61(edge)  # 执行当前语句以推进本节示例。
    return GraphBatch(x, edge, torch.zeros(n, dtype=torch.long)).validate()  # 返回当前分支计算出的结果。

def pack_graphs(graphs: list[GraphBatch]) -> GraphBatch:  # 定义本节可复用的核心函数。
    if not graphs: raise ValueError("不能打包空列表")  # 按当前条件选择后续控制路径。
    xs, edges, batches, offset = [], [], [], 0  # 计算并保存当前步骤的中间状态。
    for gid, g in enumerate(graphs):  # 遍历输入元素以累积或检查结果。
        g.validate(); validate_reciprocal_arcs61(g.edge_index)  # 执行当前语句以推进本节示例。
        xs.append(g.x); edges.append(g.edge_index + offset)  # 执行当前语句以推进本节示例。
        batches.append(torch.full((len(g.x),), gid, dtype=torch.long)); offset += len(g.x)  # 计算并保存当前步骤的中间状态。
    packed = GraphBatch(torch.cat(xs), torch.cat(edges, dim=1), torch.cat(batches)).validate()  # 计算并保存当前步骤的中间状态。
    validate_reciprocal_arcs61(packed.edge_index)  # 执行当前语句以推进本节示例。
    return packed  # 返回当前分支计算出的结果。

def graph_semantic_digest61(g: GraphBatch) -> str:  # 定义本节可复用的核心函数。
    return canonical_digest(g.semantic())  # 返回当前分支计算出的结果。

def graph_group_id61(g: GraphBatch) -> int:  # 定义本节可复用的核心函数。
    return int(graph_semantic_digest61(g)[:15], 16)  # 返回当前分支计算出的结果。

assert torch.equal(make_graph(0, 7).x, make_graph(1, 7).x)  # 用受控断言验证关键不变量。
assert not torch.equal(make_graph(0, 7).edge_index, make_graph(1, 7).edge_index)  # 用受控断言验证关键不变量。

train_variants61 = list(range(10, 22))  # 计算并保存当前步骤的中间状态。
test_variants61 = list(range(101, 107))  # 计算并保存当前步骤的中间状态。
train_labels61 = [i % 2 for i in range(len(train_variants61))]  # 计算并保存当前步骤的中间状态。
test_labels61 = [i % 2 for i in range(len(test_variants61))]  # 计算并保存当前步骤的中间状态。
train_graphs = [make_graph(label, variant) for label, variant in zip(train_labels61, train_variants61)]  # 计算并保存当前步骤的中间状态。
test_graphs = [make_graph(label, variant) for label, variant in zip(test_labels61, test_variants61)]  # 计算并保存当前步骤的中间状态。
train_digests61 = {graph_semantic_digest61(g) for g in train_graphs}  # 计算并保存当前步骤的中间状态。
test_digests61 = {graph_semantic_digest61(g) for g in test_graphs}  # 计算并保存当前步骤的中间状态。
assert len(train_digests61) == len(train_graphs) and len(test_digests61) == len(test_graphs)  # 用受控断言验证关键不变量。
assert train_digests61.isdisjoint(test_digests61)  # 用受控断言验证关键不变量。

train_group_ids61 = torch.tensor([graph_group_id61(g) for g in train_graphs], dtype=torch.long)  # 计算并保存当前步骤的中间状态。
assert train_group_ids61.unique().numel() == len(train_graphs)  # 用受控断言验证关键不变量。
train_batch = pack_graphs(train_graphs)  # 计算并保存当前步骤的中间状态。
model61 = GraphCL()  # 计算并保存当前步骤的中间状态。
opt = torch.optim.Adam(model61.parameters(), lr=0.015)  # 计算并保存当前步骤的中间状态。
history = []  # 计算并保存当前步骤的中间状态。
for step in range(24):  # 遍历输入元素以累积或检查结果。
    g1 = augment_graph(train_batch, node_drop=.05, edge_drop=.05, feature_mask=.04,  # 计算并保存当前步骤的中间状态。
                       generator=torch.Generator().manual_seed(SEED + 2*step))  # 计算并保存当前步骤的中间状态。
    g2 = augment_graph(train_batch, node_drop=.05, edge_drop=.05, feature_mask=.04,  # 计算并保存当前步骤的中间状态。
                       generator=torch.Generator().manual_seed(SEED + 2*step + 1))  # 计算并保存当前步骤的中间状态。
    loss = multi_positive_nt_xent(model61(g1), model61(g2), train_group_ids61, .25)  # 计算并保存当前步骤的中间状态。
    opt.zero_grad(); loss.backward(); opt.step(); history.append(float(loss))  # 执行当前语句以推进本节示例。

for p in model61.encoder.parameters(): p.requires_grad_(False)  # 遍历输入元素以累积或检查结果。
head61 = nn.Linear(model61.encoder.out_dim, 2)  # 计算并保存当前步骤的中间状态。
head_opt = torch.optim.Adam(head61.parameters(), lr=.05)  # 计算并保存当前步骤的中间状态。
y_train = torch.tensor(train_labels61)  # 计算并保存当前步骤的中间状态。
with torch.no_grad(): h_train = model61(train_batch, project=False)  # 在受管理的上下文中执行操作。
for _ in range(45):  # 遍历输入元素以累积或检查结果。
    supervised = F.cross_entropy(head61(h_train), y_train)  # 计算并保存当前步骤的中间状态。
    head_opt.zero_grad(); supervised.backward(); head_opt.step()  # 执行当前语句以推进本节示例。
with torch.no_grad():  # 在受管理的上下文中执行操作。
    pred = head61(model61(pack_graphs(test_graphs), project=False)).argmax(1)  # 计算并保存当前步骤的中间状态。
accuracy61 = float((pred == torch.tensor(test_labels61)).float().mean())  # 计算并保存当前步骤的中间状态。
assert math.isfinite(history[-1]) and history[-1] < history[0]  # 用受控断言验证关键不变量。
assert accuracy61 >= .99  # 用受控断言验证关键不变量。
assert all(p.grad is None or torch.isfinite(p.grad).all() for p in head61.parameters())  # 用受控断言验证关键不变量。
print({"contrastive_first": round(history[0],4), "contrastive_last": round(history[-1],4),  # 执行当前语句以推进本节示例。
       "controlled_linear_eval_accuracy": accuracy61})  # 执行当前语句以推进本节示例。


## 7. 训练、推理与复杂度边界

训练路径是 `原图 → 两个随机增强 → 共享 encoder/projector → NT-Xent`；下游推理通常丢弃 projector，只保留 encoder 表示。若每批共 $N$ 个节点、$E$ 条边、$B$ 张图，GIN 约为 $O(EH+NH^2)$；全批 NT-Xent 相似度矩阵为 $O(B^2D)$ 时间和 $O(B^2)$ 内存，大批量时需分块、cross-batch memory 或分布式 all-gather，并处理跨卡重复 ID。

常见失败包括：增强删空整张图、双 view 共用同一 RNG 状态、跨图边、把同源副本当负例、评估时仍使用 projector、按随机节点而不是按图/主体切分，以及用受控数据准确率宣称真实泛化。


## 8. 发布制品：带外 registry 而不是“自己给自己签名”

包内哈希只能发现传输损坏，攻击者替换权重后可以重算同一个哈希。下面的 `_TRUSTED_RELEASES61` 是发布系统在制品之外保存的信任锚。manifest 同时绑定模型 schema、训练/测试图语义、split、增强 recipe 和 state 的 key/dtype/shape/bytes。

`PublishedGraphCL` 也不信任调用方传入的 `graph_digest`：它在 `forward` 中从实际张量重算输入语义，只接受 manifest 声明的图。教学代码用内存 registry 模拟签名服务；生产应使用只读制品仓、KMS 签名、版本吊销和审计日志。


In [ ]:
CONFIG61 = {"in_dim":3, "hidden":16, "depth":2, "proj_dim":12}  # 计算并保存当前步骤的中间状态。
RECIPE61 = {"node_drop":.05,"edge_drop":.05,"edge_drop_unit":"reciprocal-undirected-pair","feature_mask":.04,"rng":"independent torch.Generator/seed+2*step"}  # 计算并保存当前步骤的中间状态。
SPLIT61 = {"train_variants":train_variants61,"test_variants":test_variants61,"unit":"whole_original_graph","cross_split_digest_overlap":0}  # 计算并保存当前步骤的中间状态。

def graph_digest(g: GraphBatch) -> str:  # 定义本节可复用的核心函数。
    return canonical_digest(g.semantic())  # 返回当前分支计算出的结果。

state61 = {k:v.detach().cpu().clone() for k,v in model61.state_dict().items()}  # 计算并保存当前步骤的中间状态。
manifest61 = {  # 计算并保存当前步骤的中间状态。
    "schema":"packed-x[N,3]-reciprocal-edge[2,E]-batch[N]/cross_graph_forbidden/v2",  # 执行当前语句以推进本节示例。
    "config":CONFIG61, "augment":RECIPE61, "split":SPLIT61, "positive_groups":train_group_ids61.tolist(),  # 执行当前语句以推进本节示例。
    "train_graph":graph_digest(train_batch), "test_graph":graph_digest(pack_graphs(test_graphs)),  # 执行当前语句以推进本节示例。
    "state":state_descriptor(state61),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
artifact61 = {"release_id":"graphcl-61-v1", "manifest":manifest61, "state":state61}  # 计算并保存当前步骤的中间状态。

def artifact_digest61(artifact: dict) -> str:  # 定义本节可复用的核心函数。
    return canonical_digest({"release_id":artifact["release_id"], "manifest":artifact["manifest"]})  # 返回当前分支计算出的结果。

_TRUSTED_RELEASES61 = MappingProxyType({"graphcl-61-v1": artifact_digest61(artifact61)})  # 计算并保存当前步骤的中间状态。

class PublishedGraphCL(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, model: GraphCL, allowed_graphs: tuple[str,...]):  # 定义本节可复用的核心函数。
        super().__init__(); self.model = model.eval(); self.allowed_graphs = allowed_graphs  # 计算并保存当前步骤的中间状态。
    def forward(self, graph: GraphBatch, project: bool = False) -> torch.Tensor:  # 定义本节可复用的核心函数。
        actual = graph_digest(graph)  # 从真实输入张量重算，而非接收调用者摘要
        if actual not in self.allowed_graphs:  # 按当前条件选择后续控制路径。
            raise ValueError("输入图语义不属于该 release")  # 遇到非法合同立即显式失败。
        with torch.no_grad(): return self.model(graph, project=project)  # 在受管理的上下文中执行操作。

def load_published61(artifact: dict) -> PublishedGraphCL:  # 定义本节可复用的核心函数。
    release_id = artifact.get("release_id")  # 计算并保存当前步骤的中间状态。
    if release_id not in _TRUSTED_RELEASES61 or artifact_digest61(artifact) != _TRUSTED_RELEASES61[release_id]:  # 按当前条件选择后续控制路径。
        raise ValueError("release 未受带外 registry 信任")  # 遇到非法合同立即显式失败。
    if artifact["manifest"]["state"] != state_descriptor(artifact["state"]):  # 按当前条件选择后续控制路径。
        raise ValueError("state 的 key/dtype/shape/bytes 不匹配")  # 遇到非法合同立即显式失败。
    if artifact["manifest"]["schema"] != "packed-x[N,3]-reciprocal-edge[2,E]-batch[N]/cross_graph_forbidden/v2":  # 按当前条件选择后续控制路径。
        raise ValueError("schema 不匹配")  # 遇到非法合同立即显式失败。
    cfg = artifact["manifest"]["config"]  # 计算并保存当前步骤的中间状态。
    restored = GraphCL(**cfg); restored.load_state_dict(artifact["state"], strict=True)  # 计算并保存当前步骤的中间状态。
    return PublishedGraphCL(restored, (artifact["manifest"]["train_graph"], artifact["manifest"]["test_graph"]))  # 返回当前分支计算出的结果。

published61 = load_published61(artifact61)  # 计算并保存当前步骤的中间状态。
assert published61(train_batch).shape == (12, model61.encoder.out_dim)  # 用受控断言验证关键不变量。
tampered61 = copy.deepcopy(artifact61)  # 计算并保存当前步骤的中间状态。
tampered61["state"][next(iter(tampered61["state"]))].view(-1)[0] += 1  # 计算并保存当前步骤的中间状态。
tampered61["manifest"]["state"] = state_descriptor(tampered61["state"])  # 计算并保存当前步骤的中间状态。
try:  # 尝试执行可能失败的受控操作。
    load_published61(tampered61)  # 执行当前语句以推进本节示例。
    raise AssertionError("整体重签攻击未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "registry" in str(exc)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    published61(make_graph(0, 99))  # 执行当前语句以推进本节示例。
    raise AssertionError("未发布输入语义未被拒绝")  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert "输入图语义" in str(exc)  # 用受控断言验证关键不变量。


## 9. 工程检查清单与生产差距

上线前至少补齐：真实领域增强的因果合理性；增强强度消融；图/主体/时间级隔离；跨卡正例 ID 对齐；大 batch 相似度分块；近重复假负例审计；多 seed 置信区间；OOM/空图/超大图降级；权重签名与回滚；输入 schema、特征词典和预处理版本监控。

本例没有声称“GraphCL 在任意数据上有效”。它验证的是：手写算子遵循置换与隔离合同，多正例目标数值正确，随机增强可复现，受控训练能学习，并且发布加载不把调用方自签摘要当成信任来源。


In [ ]:
assert len(_TRUSTED_RELEASES61) == 1  # 用受控断言验证关键不变量。
assert set(manifest61) == {"schema","config","augment","split","positive_groups","train_graph","test_graph","state"}  # 用受控断言验证关键不变量。
assert accuracy61 == 1.0  # 用受控断言验证关键不变量。
print("GraphCL 61：所有数学、隔离、训练与发布 oracle 通过。")  # 执行当前语句以推进本节示例。
